# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [4]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [8]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
data = {
                "table": [
                        "account",
                        "client",
                        "disposition",
                        "district",
                        "loan",
                        "order",
                        "trans",
                        "card",
                ],
                "definition": [
                        "Bank accounts, including account ID, district, creation date, and statement frequency.",
                        "Clients, including client ID, district, birth date, and gender.",
                        "Relationships between clients and accounts, including the relationship type.",
                        "Districts and their demographic and economic statistics.",
                        "Loans, including amount, duration, payment amount, date, and repayment status.",
                        "Standing orders, including the account, recipient, amount, and payment type.",
                        "Account transactions, including date, amount, type, and related account.",
                        "Payment cards, including card type, issue date, and associated disposition.",
                ],
        }

df = pd.DataFrame(data)
print(df)



         table                                         definition
0      account  Bank accounts, including account ID, district,...
1       client  Clients, including client ID, district, birth ...
2  disposition  Relationships between clients and accounts, in...
3     district  Districts and their demographic and economic s...
4         loan  Loans, including amount, duration, payment amo...
5        order  Standing orders, including the account, recipi...
6        trans  Account transactions, including date, amount, ...
7         card  Payment cards, including card type, issue date...


In [10]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [11]:
print(text_tables)

account: Bank accounts, including account ID, district, creation date, and statement frequency.
client: Clients, including client ID, district, birth date, and gender.
disposition: Relationships between clients and accounts, including the relationship type.
district: Districts and their demographic and economic statistics.
loan: Loans, including amount, duration, payment amount, date, and repayment status.
order: Standing orders, including the account, recipient, amount, and payment type.
trans: Account transactions, including date, amount, type, and related account.
card: Payment cards, including card type, issue date, and associated disposition.


In [13]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [15]:
#Creating the prompt, with the user questions and the tables definitions.
user_question = "Which clients have loans, and what are the loan amounts and repayment statuses?"
pqt1 = prompt_question_tables.format(tables=text_tables, question=user_question)

In [16]:
print(return_OAI(pqt1))

{
    "tables": ["client", "loan"]
}


In [18]:
pqt3 = prompt_question_tables.format(
    tables=text_tables,
    question="Which accounts have payment cards, and what type of card does each account have?"
)

In [19]:
print(return_OAI(pqt3))

```json
{
    "tables": ["account", "card"]
}
```


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [20]:

prompt_question_tables_v2 = """
You are an assistant that selects database tables for SQL queries.

Available tables and their definitions:
{tables}

User question:
{question}

Instructions:
1. Identify the tables required to answer the question.
2. Use only the table names listed above.
3. Select the minimum set of tables needed.
4. Consider relationships between tables when selecting them.
5. Do not generate SQL or invent columns.
6. Return valid JSON only, with this structure:
{{"tables": ["table_name_1", "table_name_2"]}}
"""

def return_OAI(user_message):
    client = OpenAI(api_key=OPENAI_API_KEY)

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {
                "role": "system",
                "content": "Select database tables carefully and follow the requested output format."
            },
            {
                "role": "user",
                "content": user_message
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content


In [21]:

import json

test_questions = [
    "Which clients have loans, and what are the loan amounts and repayment statuses?",
    "Which accounts have payment cards, and what type of card does each account have?",
    "How many clients live in each district?",
    "What are the total outgoing transaction amounts for each account?",
    "Which clients have accounts and what is the balance of each account?",
    "Which districts have the highest number of loans?"
]

for question in test_questions:
    prompt = prompt_question_tables_v2.format(
        tables=text_tables,
        question=question
    )

    result = return_OAI(prompt)

    print("\nQuestion:", question)
    print("Model response:", result)

    try:
        parsed = json.loads(result)
        print("Selected tables:", parsed["tables"])
    except (json.JSONDecodeError, KeyError, TypeError):
        print("Warning: Response was not valid JSON in the expected format.")



Question: Which clients have loans, and what are the loan amounts and repayment statuses?
Model response: 
{"tables": ["client", "loan"]}
Selected tables: ['client', 'loan']

Question: Which accounts have payment cards, and what type of card does each account have?
Model response: {
	"tables": ["account", "card"]
}
Selected tables: ['account', 'card']

Question: How many clients live in each district?
Model response: 
{"tables": ["client", "district"]}
Selected tables: ['client', 'district']

Question: What are the total outgoing transaction amounts for each account?
Model response: {"tables": ["trans"]}
Selected tables: ['trans']

Question: Which clients have accounts and what is the balance of each account?
Model response: {
	"tables": ["client", "account", "trans"]
}
Selected tables: ['client', 'account', 'trans']

Question: Which districts have the highest number of loans?
Model response: 
{
	"tables": ["district", "loan"]
}
Selected tables: ['district', 'loan']


In [22]:

prompt_question_tables_v3 = """
Choose the database tables needed to answer the question.

Available tables:
{tables}

Examples:
Question: Which clients have loans?
Tables: ["client", "disposition", "loan"]

Question: What types of cards exist?
Tables: ["card"]

Now answer this question:
{question}

Return only a JSON object in this format:
{{"tables": ["table1", "table2"]}}

Use only available table names. Do not invent tables.
"""

question = "How many clients live in each district?"

prompt = prompt_question_tables_v3.format(
    tables=text_tables,
    question=question
)

print(return_OAI(prompt))


{"tables": ["client", "district"]}


# SQL Query from Table Names — Experiment Report

## Lessons Learned

I learned that prompt design has a significant role in guiding an LLM to perform structured tasks. Explicit constraints and examples can improve consistency, while low temperature helps reduce response variability.

I also learned that identifying the correct tables is only the first step in generating SQL. A reliable system must understand table relationships, validate column names, and handle questions that cannot be answered using the available schema.

Overall, LLMs can assist with database question answering, but their table selections should be validated before generating or executing SQL queries.
